# EXERCISE 4: TF-IDF COMPUTATION FROM SCRATCH & VISUALIZATION
### Aim:
To compute Term Frequency-Inverse Document Frequency (TF-IDF) step-by-step from mathematical first principles:
1. Tokenization and Vocabulary Construction
2. Term Frequency (TF) Matrix
3. Document Frequency (DF) and Inverse Document Frequency (IDF)
4. TF-IDF Matrix formulation and Heatmap Visualization

In [1]:
import math
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Corpus Documents
documents = [
    "Cloud computing provides scalable storage",
    "Cloud computing delivers reliable servers",
    "Quantum computing enables fast storage"
]

document_names = ["D1", "D2", "D3"]

corpus_df = pd.DataFrame({
    "Document": document_names,
    "Text": documents
})

print("Document Corpus:")
print(corpus_df.to_string(index=False))

Document Corpus:
Document                                     Text
      D1 Cloud computing provides scalable storage
      D2 Cloud computing delivers reliable servers
      D3    Quantum computing enables fast storage


In [3]:
def tokenize(text):
    text = text.lower()
    return re.findall(r"\b[a-z]+\b", text)

tokenized_documents = [tokenize(doc) for doc in documents]

print("Tokenized Documents:")
for name, tokens in zip(document_names, tokenized_documents):
    print(f"{name}: {tokens}")

Tokenized Documents:
D1: ['cloud', 'computing', 'provides', 'scalable', 'storage']
D2: ['cloud', 'computing', 'delivers', 'reliable', 'servers']
D3: ['quantum', 'computing', 'enables', 'fast', 'storage']



In [4]:
vocabulary = sorted(set(
    word
    for tokens in tokenized_documents
    for word in tokens
))

print("Vocabulary:")
print(vocabulary)
print("\nVocabulary size:", len(vocabulary))

Vocabulary:
['cloud', 'computing', 'delivers', 'enables', 'fast', 'provides', 'quantum', 'reliable', 'scalable', 'servers', 'storage']

Vocabulary size: 11


In [5]:
word_counts = {
    name: len(tokens)
    for name, tokens in zip(document_names, tokenized_documents)
}

word_count_df = pd.DataFrame(
    list(word_counts.items()),
    columns=["Document", "Total Words"]
)

print("Document Word Counts:")
print(word_count_df.to_string(index=False))

Document Word Counts:
Document  Total Words
      D1            5
      D2            5
      D3            5


In [6]:
# TERM FREQUENCY (TF) MATRIX
# TF(t, d) = count(t in d) / total words in d

tf_matrix = pd.DataFrame(
    0.0,
    index=document_names,
    columns=vocabulary
)

for doc_name, tokens in zip(document_names, tokenized_documents):
    total_words = len(tokens)
    for term in vocabulary:
        count = tokens.count(term)
        tf_matrix.loc[doc_name, term] = count / total_words

print("Term Frequency (TF) Matrix:")
print(tf_matrix.round(4))

Term Frequency (TF) Matrix:
    cloud  computing  delivers  enables  ...  reliable  scalable  servers  storage
D1    0.2        0.2       0.0      0.0  ...       0.0       0.2      0.0      0.2
D2    0.2        0.2       0.2      0.0  ...       0.2       0.0      0.2      0.0
D3    0.0        0.2       0.0      0.2  ...       0.0       0.0      0.0      0.2

[3 rows x 11 columns]


In [7]:
plt.figure(figsize=(10, 4))
sns.heatmap(tf_matrix, annot=True, fmt=".3f", cmap="Blues")
plt.title("Term Frequency (TF) Heatmap")
plt.xlabel("Terms")
plt.ylabel("Documents")
plt.show()

In [8]:
# DOCUMENT FREQUENCY (DF)
df_values = {}
for term in vocabulary:
    df_values[term] = sum(
        1 for tokens in tokenized_documents
        if term in tokens
    )

df_series = pd.Series(df_values, name="DF")

print("Document Frequency (DF):")
print(df_series)

Document Frequency (DF):
cloud        2
computing    3
delivers     1
enables      1
fast         1
provides     1
quantum      1
reliable     1
scalable     1
servers      1
storage      2
Name: DF, dtype: int64


In [9]:
# INVERSE DOCUMENT FREQUENCY (IDF)
# Smoothed formula: IDF(t) = ln((N + 1) / (DF(t) + 1)) + 1
N = len(documents)

idf_values = {
    term: math.log((N + 1) / (df_values[term] + 1)) + 1
    for term in vocabulary
}

idf_series = pd.Series(idf_values, name="IDF")

print("Inverse Document Frequency (IDF):")
print(idf_series.round(4))

Inverse Document Frequency (IDF):
cloud        1.2877
computing    1.0000
delivers     1.6931
enables      1.6931
fast         1.6931
provides     1.6931
quantum      1.6931
reliable     1.6931
scalable     1.6931
servers      1.6931
storage      1.2877
Name: IDF, dtype: float64


In [10]:
# TF-IDF MATRIX = TF * IDF
tfidf_raw = tf_matrix.mul(idf_series, axis="columns")

print("Raw TF-IDF Matrix:")
print(tfidf_raw.round(4))

Raw TF-IDF Matrix:
     cloud  computing  delivers  enables  ...  reliable  scalable  servers  storage
D1  0.2575        0.2    0.0000   0.0000  ...    0.0000    0.3386   0.0000   0.2575
D2  0.2575        0.2    0.3386   0.0000  ...    0.3386    0.0000   0.3386   0.0000
D3  0.0000        0.2    0.0000   0.3386  ...    0.0000    0.0000   0.0000   0.2575

[3 rows x 11 columns]


In [11]:
plt.figure(figsize=(10, 4))
sns.heatmap(tfidf_raw, annot=True, fmt=".3f", cmap="Greens")
plt.title("Raw TF-IDF Matrix Heatmap")
plt.xlabel("Terms")
plt.ylabel("Documents")
plt.show()

### Interpretation:
1. Common words across documents such as **"computing"** appear in all 3 documents ($DF=3$), giving it the lowest IDF score ($1.0000$).
2. Distinctive, discriminative words such as **"scalable"**, **"reliable"**, **"quantum"**, and **"servers"** appear in only one document ($DF=1$), resulting in the highest IDF weights ($1.6931$).
3. The TF-IDF matrix balances term prominence with uniqueness, assigning higher weights to informative terms while diminishing common words.